In [ ]:
import re
import pandas as pd
from collections import Counter

# ==========================================================
# CONFIGURATION
# ==========================================================

INPUT_CSV = "stage1.csv" # Run each stage csv file individually
OUTPUT_CSV = "relative_frequencies.csv" # Rename it as you wish

# ==========================================================
# PRONOUN REGEX
# ==========================================================

PRONOUN_PATTERN = re.compile(
    r'([ON])'                  # O / N
    r'([123]P)'                # Person
    r'(Sg|Pl)'                # Number
    r'([+-]R)?'                # Referentiality
    r'([+-]I)?'                # Weak "it"
    r'([+-]A)?'                # Referential anomaly
    r'(Exp|Gen)?'              # Expletive / Generic
    r'(SUB|OBJ|POSS|PREP)?'    # Role
    r'(SUB|OBJ|POSS|PREP)?'    # Role duplication because we may have tags like 'POSSPREP'
    r'(MS|ES|CS)?'             # Clause
)

# ==========================================================
# SENTENCE REGEX
# ==========================================================

SENTENCE_PATTERN = re.compile(
    r'\['              # opening bracket
    r'(MS|ES|CS)'      # sentence type
    r'(TRUNC)?'        # optional TRUNC
    r'([+-]A)?'        # optional anomaly
)

# ==========================================================
# PARSE PRONOUNS
# ==========================================================

def parse_pronouns(text):

    tokens = []

    for m in PRONOUN_PATTERN.finditer(str(text)):

        token = {
            "Realization": m.group(1),
            "Person": m.group(2),
            "Number": m.group(3),
            "Referentiality": m.group(4),
            "WeakIt": m.group(5),
            "Anomaly": m.group(6),
            "Special": m.group(7),
            "Role1": m.group(8),
            "Role2": m.group(9),
            "Clause": m.group(10),
        }

        tokens.append(token)

    return tokens

# ==========================================================
# PARSE SENTENCES
# ==========================================================

def parse_sentences(text):

    sentences = []

    for m in SENTENCE_PATTERN.finditer(str(text)):

        sentence = {
            "SentenceType": m.group(1),
            "Truncated": m.group(2),
            "Anomaly": m.group(3),
        }

        sentences.append(sentence)

    return sentences

# ==========================================================
# FEATURE ORDER (PRONOUNS)
# ==========================================================

PRONOUN_ORDER = [
    "Realization",
    "Person",
    "Number",
    "Referentiality",
    "WeakIt",
    "Anomaly",
    "Special",
    "Role1",
    "Role2",
    "Clause",
]

# ==========================================================
# GENERATE PRONOUN FEATURES
# ==========================================================

def pronoun_features(token):

    features = []

    current = ""

    for key in PRONOUN_ORDER:

        value = token[key]

        if value is None:
            continue

        current += value

        features.append(current)

    return features

# ==========================================================
# GENERATE SENTENCE FEATURES
# ==========================================================

def sentence_features(sentence):

    features = []

    # MS / ES / CS

    current = sentence["SentenceType"]

    features.append(current)

    # MSTRUNC / ESTRUNC

    if sentence["Truncated"] is not None:

        current += sentence["Truncated"]

        features.append(current)

    # MSTRUNC+A

    if sentence["Anomaly"] is not None:

        current += sentence["Anomaly"]

        features.append(current)

    return features

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv(INPUT_CSV)

rows = []

# ==========================================================
# PROCESS EACH ROW
# ==========================================================

for _, row in df.iterrows():

    counts = Counter()

    #######################################################
    # PRONOUNS
    #######################################################

    pronouns = parse_pronouns(row["analysis"])

    for token in pronouns:

        for feature in pronoun_features(token):

            counts[feature] += 1

    #######################################################
    # SENTENCES
    #######################################################

    sentences = parse_sentences(row["analysis"])

    for sentence in sentences:

        for feature in sentence_features(sentence):

            counts[feature] += 1

        # Overall truncation count

        if sentence["Truncated"] is not None:

            counts["TRUNC"] += 1

        # Overall anomalous truncations

        if sentence["Truncated"] is not None and sentence["Anomaly"] is not None:

            counts["TRUNC" + sentence["Anomaly"]] += 1

    #######################################################
    # NORMALIZE BY WORD COUNT
    #######################################################

    wc = row["word_count_analyzed"]

    relative = {}

    for feature, count in counts.items():

        relative[feature] = count / wc

    # Copy every original column
    new_row = row.to_dict()

    # Add every computed relative frequency
    new_row.update(relative)

    rows.append(new_row)

# ==========================================================
# OUTPUT
# ==========================================================

result = pd.DataFrame(rows)

# Any frequency that did not occur becomes 0.
# Original columns remain unchanged because they are already populated.
frequency_columns = result.columns.difference(df.columns)

result[frequency_columns] = result[frequency_columns].fillna(0)

result.to_csv(OUTPUT_CSV, index=False)

print(result.head())

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, shapiro

# ==========================================================
# CONFIGURATION
# ==========================================================

INPUT_CSV = "relative_frequencies.csv" # This is the output of the previous script, run this cell for each stage

NORMALITY_OUTPUT = "normality_results.csv"
MW_OUTPUT = "mann_whitney_results.csv"
SIGNIFICANT_OUTPUT = "significant_results.csv"

GROUP_COLUMN = 0        # first column
FIRST_TAG_COLUMN = 11   # first frequency variable (0-indexed)

# ==========================================================
# LOAD DATA
# ==========================================================

df = pd.read_csv(INPUT_CSV)

group_column = df.columns[GROUP_COLUMN]
tag_columns = df.columns[FIRST_TAG_COLUMN:]

group0 = df[df[group_column] == 0]
group1 = df[df[group_column] == 1]

# ==========================================================
# NORMALITY
# ==========================================================

normality_results = []

for variable in tag_columns:

    x0 = group0[variable].dropna()
    x1 = group1[variable].dropna()

    # Shapiro only works with at least 3 observations
    if len(x0) >= 3:
        stat0, p0 = shapiro(x0)
    else:
        stat0, p0 = np.nan, np.nan

    if len(x1) >= 3:
        stat1, p1 = shapiro(x1)
    else:
        stat1, p1 = np.nan, np.nan

    normality_results.append({
        "Variable": variable,
        "Group0_W": stat0,
        "Group0_p": p0,
        "Group1_W": stat1,
        "Group1_p": p1
    })

normality_df = pd.DataFrame(normality_results)
normality_df.to_csv(NORMALITY_OUTPUT, index=False)

# ==========================================================
# MANN-WHITNEY
# ==========================================================

mw_results = []

for variable in tag_columns:

    x0 = group0[variable].dropna()
    x1 = group1[variable].dropna()

    if len(x0) == 0 or len(x1) == 0:
        continue

    try:

        U, p = mannwhitneyu(
            x0,
            x1,
            alternative="two-sided"
        )

        n0 = len(x0)
        n1 = len(x1)

        # Rank-biserial correlation
        rank_biserial = (2 * U) / (n0 * n1) - 1

    except Exception:

        U = np.nan
        p = np.nan
        rank_biserial = np.nan

    mw_results.append({

        "Variable": variable,

        "U": U,
        "Rank_Biserial": rank_biserial,
        "p": p,

        "Mean_Group0": x0.mean(),
        "SD_Group0": x0.std(ddof=1),

        "Mean_Group1": x1.mean(),
        "SD_Group1": x1.std(ddof=1),

        "Median_Group0": x0.median(),
        "Median_Group1": x1.median(),

        "N_Group0": len(x0),
        "N_Group1": len(x1)

    })

mw_df = pd.DataFrame(mw_results)

mw_df = mw_df.sort_values("p")

mw_df.to_csv(MW_OUTPUT, index=False)

# ==========================================================
# SIGNIFICANT VARIABLES
# ==========================================================

significant = mw_df[mw_df["p"] < 0.05].copy()

# Formatted descriptive statistics
significant["Group0"] = significant.apply(
    lambda r: f"{r['Mean_Group0']:.4f} ({r['SD_Group0']:.4f})",
    axis=1
)

significant["Group1"] = significant.apply(
    lambda r: f"{r['Mean_Group1']:.4f} ({r['SD_Group1']:.4f})",
    axis=1
)

# Select columns for final output
significant = significant[[
    "Variable",
    "U",
    "Rank_Biserial",
    "p",

    "Mean_Group0",
    "SD_Group0",

    "Mean_Group1",
    "SD_Group1",

    "Group0",
    "Group1",

    "Median_Group0",
    "Median_Group1",

    "N_Group0",
    "N_Group1"
]]

significant.to_csv(SIGNIFICANT_OUTPUT, index=False)

# ==========================================================
# SUMMARY
# ==========================================================

print("Finished!")

print(f"Normality results: {NORMALITY_OUTPUT}")
print(f"Mann-Whitney results: {MW_OUTPUT}")
print(f"Significant variables: {SIGNIFICANT_OUTPUT}")

print(f"\nVariables analyzed: {len(tag_columns)}")
print(f"Significant variables (p < 0.05): {len(significant)}")